<h1>Feature Engineering</h1>
<hr/>
<p>This notebook focuses on creating new features that may help the regression model better understand housing price patterns. The explanation style follows the original notebook by explaining what is being changed, why it is being changed, and how the result is verified.</p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing Required Modules</h1>
<hr/>
<p>The feature engineering workflow uses reusable code from the project package. This keeps the notebook readable while ensuring that the same logic is used during production model training.</p>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import pandas as pd

from us_housing_price_prediction.data import load_housing_data, split_features_target
from us_housing_price_prediction.features import HousingFeatureEngineer

pd.set_option("display.float_format", "{:.3f}".format)


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Separating Features and Target</h1>
<hr/>
<p>Before feature engineering is conducted, the target variable must be separated from the input features. This is important because the target column must never be used as an input feature. The identifier column is also removed because it does not contribute meaningful predictive information.</p>


In [ ]:
df = load_housing_data()
X, y = split_features_target(df)

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")
X.head()


<p>With reference to the dataframe above, the model features contain only property attributes. The target price is stored separately, which prevents target leakage during modeling.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Creating New Features</h1>
<hr/>
<p>Feature engineering is used to create new columns that may represent housing characteristics more effectively than the raw columns alone. The engineered features used in this project are listed below.</p>
<br/>
<ul>
    <li><strong>area_per_bedroom</strong>: House area divided by number of bedrooms.</li>
    <li><strong>toilet_to_bedroom_ratio</strong>: Number of toilets divided by number of bedrooms.</li>
    <li><strong>total_rooms</strong>: Bedrooms plus toilets.</li>
    <li><strong>stories_house_area_interaction</strong>: Stories multiplied by house area.</li>
</ul>


In [ ]:
engineer = HousingFeatureEngineer(drop_source_features=True)
X_engineered = engineer.fit_transform(X)
X_engineered.head(10)


<p>From the output above, the newly engineered features have been created successfully. These features help represent the relationship between house size, room count, and number of stories in a more informative way.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Removing Redundant Source Features</h1>
<hr/>
<p>After feature engineering, some raw source features can become redundant because their information is already represented in the engineered features. Keeping all of them may increase repeated signal and make model interpretation less clean.</p>
<br/>
<p>One improvement made in this project is to drop selected source fields after the derived features are created. This directly addresses the feature engineering issue where old features were not removed after stronger features had been introduced.</p>


In [ ]:
removed_columns = sorted(set(X.columns) - set(X_engineered.columns))
added_columns = sorted(set(X_engineered.columns) - set(X.columns))

pd.DataFrame(
    {
        "removed_after_engineering": pd.Series(removed_columns),
        "added_engineered_features": pd.Series(added_columns),
    }
)


<p>With reference to the table above, the workflow clearly documents which columns were added and which source columns were removed. This makes the feature engineering process more transparent and easier to justify.</p>


<hr/>
<h1>5.&nbsp;&nbsp;&nbsp;&nbsp;Reviewing Engineered Data</h1>
<hr/>
<p>The final engineered dataframe should be checked to ensure that the new features are numeric and that the categorical columns remain available for encoding inside the pipeline.</p>


In [ ]:
X_engineered.info()


In [ ]:
X_engineered.describe(include="all").T


<p>With reference to the summary above, the engineered numerical features can be scaled and imputed, while categorical features can be one-hot encoded later in the model pipeline. This keeps preprocessing consistent and prevents leakage.</p>
